In [8]:
import os
import re
import math
import json
import logging
import random
import numpy as np
import pandas as pd
import scipy.linalg
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms, models
from typing import Optional
from torchvision.models import inception_v3, Inception_V3_Weights
from scipy.optimize import linear_sum_assignment

In [9]:
# -------------------------------
# 設定 logging 等級，方便印出訓練與評估時的訊息
# -------------------------------
logging.basicConfig(level=logging.INFO)

# -------------------------------
# 常數定義：時間嵌入維度
# -------------------------------
TIME_EMB_DIM = 128

# --------------------------------------
# 數據處理相關
# --------------------------------------
def parse_lat_lon(column_name: str) -> tuple[float, float]:
    """
    解析欄位名稱中的經緯度資訊，假設格式為 "name (lon, lat)"。

    參數:
        column_name: 欄位名稱，必須包含以括號包住的經緯度資訊，例如 "(121.565, 25.033)"。

    回傳:
        一個元組 (經度, 緯度) 的浮點數。

    若格式不正確則拋出 ValueError。
    """
    match = re.search(r'\(([\d.-]+),\s*([\d.-]+)\)', column_name)
    if match:
        return float(match.group(1)), float(match.group(2))
    raise ValueError(f"欄位名稱格式無效：{column_name}")

class PeopleFlowDatasetCondition(Dataset):
    def __init__(self, csv_path: str, H: int, W: int, condition_length: int,
                 prediction_length: int, transform: Optional[callable] = None,
                 normalize: bool = True, debug: bool = False):
        if not os.path.exists(csv_path):
            raise FileNotFoundError(f"CSV 檔案未找到：{csv_path}")

        self.df = pd.read_csv(csv_path)
        self.transform = transform
        self.condition_length = condition_length
        self.prediction_length = prediction_length
        self.total_length = condition_length + prediction_length
        self.normalize_flag = normalize
        self.H, self.W = H, W

        # 1. 提取所有可用的經緯度座標及其原始欄位名
        all_flow_columns_with_coords = [c for c in self.df.columns if '(' in c and ')' in c]
        
        num_required_points = H * W
        if len(all_flow_columns_with_coords) < num_required_points:
            raise ValueError(
                f"網格大小 ({H}x{W}={num_required_points}) 大於了可用的地理座標點數量 ({len(all_flow_columns_with_coords)})."
                " 請減少 H*W 或提供更多座標點。"
            )
        
        # 解析所有座標點
        all_column_info = [] # 存儲 (原始欄位名, lon, lat, 原始索引)
        for original_idx, col_name in enumerate(all_flow_columns_with_coords):
            lon, lat = parse_lat_lon(col_name)
            all_column_info.append({'name': col_name, 'lon': lon, 'lat': lat, 'original_idx': original_idx})
        
        all_coords_np = np.array([(info['lon'], info['lat']) for info in all_column_info])

        # 如果座標點多於網格數，選擇最靠近幾何中心的 num_required_points 個點
        if len(all_column_info) > num_required_points:
            print(f"訊息: 座標點數量 ({len(all_column_info)}) 多於網格數 ({num_required_points}). "
                  f"將選擇最靠近地理中心的 {num_required_points} 個座標點進行映射。")
            
            # 計算所有點的幾何中心
            geometric_center_lon = np.mean(all_coords_np[:, 0])
            geometric_center_lat = np.mean(all_coords_np[:, 1])
            
            # 計算每個點到幾何中心的距離
            distances_to_geometric_center = np.sqrt(
                (all_coords_np[:, 0] - geometric_center_lon)**2 +
                (all_coords_np[:, 1] - geometric_center_lat)**2
            )
            
            # 獲取距離最近的 num_required_points 個點的索引 (相對於 all_coords_np)
            selected_indices_in_all_coords = np.argsort(distances_to_geometric_center)[:num_required_points]
            
            # 更新 self.column_info 和 real_coords_np 只包含選中的點
            self.column_info = [all_column_info[i] for i in selected_indices_in_all_coords]
            real_coords_np = all_coords_np[selected_indices_in_all_coords]
            
        elif len(all_column_info) == num_required_points:
            self.column_info = all_column_info
            real_coords_np = all_coords_np
        else: 
            # 這種情況已在上面檢查過 len(all_flow_columns_with_coords) < num_required_points 時拋出錯誤
            # 但為了代碼完整性可以保留
            pass 


        # --- 後續的匈牙利算法分配邏輯與之前相同，使用 self.column_info 和 real_coords_np ---
        # 2. 計算網格的理論目標地理中心
        overall_mean_lon, overall_mean_lat = np.mean(real_coords_np, axis=0)
        grid_center_lon, grid_center_lat = overall_mean_lon, overall_mean_lat

        unique_lons = np.unique(real_coords_np[:, 0])
        unique_lats = np.unique(real_coords_np[:, 1])
        lon_diffs = np.diff(np.sort(unique_lons))
        lat_diffs = np.diff(np.sort(unique_lats))
        
        lon_step = np.median(lon_diffs[lon_diffs > 0]) if len(lon_diffs[lon_diffs > 0]) > 0 else 0.005
        lat_step = np.median(lat_diffs[lat_diffs > 0]) if len(lat_diffs[lat_diffs > 0]) > 0 else 0.005
        if lon_step == 0: lon_step = 0.005
        if lat_step == 0: lat_step = 0.005

        grid_target_coords = np.zeros((num_required_points, 2))
        grid_idx_to_rc_map = {}
        current_grid_idx = 0
        for r in range(H):
            for c in range(W):
                target_lon = grid_center_lon + (c - (W - 1) / 2.0) * lon_step
                target_lat = grid_center_lat - (r - (H - 1) / 2.0) * lat_step
                grid_target_coords[current_grid_idx, 0] = target_lon
                grid_target_coords[current_grid_idx, 1] = target_lat
                grid_idx_to_rc_map[current_grid_idx] = (r, c)
                current_grid_idx += 1
        
        # 3. 計算成本矩陣
        cost_matrix = np.zeros((num_required_points, num_required_points))
        for i in range(num_required_points):
            for j in range(num_required_points):
                dist_sq = (real_coords_np[i, 0] - grid_target_coords[j, 0])**2 + \
                          (real_coords_np[i, 1] - grid_target_coords[j, 1])**2
                cost_matrix[i, j] = np.sqrt(dist_sq)

        # 4. 使用匈牙利算法
        assigned_real_coord_indices, assigned_grid_indices = linear_sum_assignment(cost_matrix)
        
        # 5. 構建最終的網格
        final_grid_assignment = np.full((H, W), -1, dtype=int) # 存儲 real_coords_np 中的索引
        
        temp_assignment_map = {grid_idx: real_idx for real_idx, grid_idx in zip(assigned_real_coord_indices, assigned_grid_indices)}
        
        flat_grid_indices_in_order = [] # 按照 (0,0)...(H-1,W-1) 順序排列的、分配到這些網格的 real_coords_np 索引

        for grid_1d_idx in range(num_required_points):
            r_map, c_map = grid_idx_to_rc_map[grid_1d_idx]
            # real_coord_idx_for_this_grid 是 temp_assignment_map 的 value, 它是 real_coords_np 的索引
            real_coord_idx_for_this_grid = temp_assignment_map[grid_1d_idx]
            final_grid_assignment[r_map, c_map] = real_coord_idx_for_this_grid # 這裡存的是 real_coords_np 的索引
            flat_grid_indices_in_order.append(real_coord_idx_for_this_grid)

        # self.sorted_flow_columns 應使用 self.column_info (它現在只包含選中的點)
        # flat_grid_indices_in_order 中的索引是相對於 real_coords_np 和 self.column_info 的
        self.sorted_flow_columns = [self.column_info[idx]['name'] for idx in flat_grid_indices_in_order]


        self._plot_grid(save_path=r"C:\thesis\code\result_ddpm\plot_grid_hungarian_centered_selection.png")

        # 6. 根據排序好的欄位順序取出 CSV 中的數據
        flow_values = self.df[self.sorted_flow_columns].values.reshape(-1, H, W).astype(np.float32)
        self.data = torch.from_numpy(flow_values)
        
        if self.normalize_flag:
            self.mean_val = self.data.mean()
            self.std_val = self.data.std() + 1e-5
            self.data = (self.data - self.mean_val) / self.std_val
        
        self.max_index = self.data.shape[0] - self.total_length + 1

    def _plot_grid(self, save_path: str):
        """
        繪製網格圖，顯示每個網格點對應的經緯度及其座標位置，
        並將結果存檔到指定的路徑。
        """
        locations = [parse_lat_lon(col) for col in self.sorted_flow_columns]
        longitudes, latitudes = zip(*locations)
        plt.figure(figsize=(12, 12))
        plt.scatter(longitudes, latitudes, c='blue', marker='o', label='Grid Points')
        # 在每個點旁顯示網格索引
        for i in range(self.H):
            for j in range(self.W):
                idx = i * self.W + j
                plt.text(longitudes[idx], latitudes[idx], f'[{i},{j}]', fontsize=6, ha='right')
        plt.xlabel("Longitude")
        plt.ylabel("Latitude")
        plt.title("Grid Arrangement")
        plt.grid(True)
        plt.legend()
        plt.savefig(save_path, dpi=600, bbox_inches='tight', pad_inches=0.1)
        plt.close()

    def __len__(self) -> int:
        # 數據集的總樣本數為可滑動視窗的數量
        return self.max_index

    def __getitem__(self, idx):
        """
        根據給定的索引，返回模型輸入與目標序列。
        輸出:
            model_input: 包含條件序列與目標序列的合併結果，形狀 (1, condition_length+prediction_length, H, W)
            target_seq: 真實的目標序列，形狀 (1, 1, H, W)
        """
        cond_seq = self.data[idx:idx + self.condition_length]  # (8, 21, 21)
        target_seq = self.data[idx + self.condition_length:idx + self.total_length]  # (1, 21, 21)
        # 將條件序列與目標序列沿著時間維度連接，並加上 batch 維度
        model_input = torch.cat([cond_seq, target_seq], dim=0).unsqueeze(0)  # (9, 21, 21) -> (1, 9, 21, 21)
        return model_input, target_seq.unsqueeze(0)  # 返回 (1, 9, 21, 21) 與 (1, 1, 21, 21)

def collate_fn(batch):
    """
    自訂批次處理函數，用於 DataLoader。
    將每個樣本的 model_input 與 target 分別堆疊成 batch。
    """
    conds, targets = zip(*batch)
    return torch.stack(conds), torch.stack(targets)

In [10]:
# --------------------------------------
# 模型定義
# --------------------------------------
class DoubleConv3D(nn.Module):
    """
    定義 3D 卷積層組合，包含兩次卷積、BatchNorm 與 ReLU 激活函數。
    此結構常用於 U-Net 中作為基本模組。
    """
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.conv = nn.Sequential(
            # 第一次卷積
            nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True),
            # 第二次卷積
            nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.conv(x)

class UNet3D(nn.Module):
    """
    3D U-Net 結構，包含下採樣（Encoder）、中間瓶頸層與上採樣（Decoder）。
    此模型同時接收噪聲版本的目標數據 x_t、完整序列 x_full 與時間嵌入。
    """
    def __init__(self, in_channels=1, base_channels=64, time_emb_dim=128, dropout_rate=0.0):
        super().__init__()
        # 編碼器部分：逐層進行雙卷積與下採樣
        self.enc1 = DoubleConv3D(in_channels, base_channels)
        self.pool1 = nn.MaxPool3d((2, 2, 2))
        self.enc2 = DoubleConv3D(base_channels, base_channels * 2)
        self.pool2 = nn.MaxPool3d((2, 2, 2))
        self.enc3 = DoubleConv3D(base_channels * 2, base_channels * 4)
        self.pool3 = nn.MaxPool3d((2, 2, 2))
        self.enc4 = DoubleConv3D(base_channels * 4, base_channels * 8)
        # 這裡使用不同的池化參數以調整深度與空間尺寸
        self.pool4 = nn.MaxPool3d(kernel_size=(2, 2, 2), stride=(1, 2, 2), padding=(1, 0, 0))
        # 瓶頸層
        self.bottleneck = DoubleConv3D(base_channels * 8, base_channels * 16)
        # 解碼器部分：逐層上採樣並與對應編碼層做 concat
        self.up4 = nn.ConvTranspose3d(base_channels * 16, base_channels * 8, kernel_size=(2, 2, 2), stride=(1, 2, 2), output_padding=(0, 1, 1))
        self.dec4 = DoubleConv3D(base_channels * 16, base_channels * 8)
        self.up3 = nn.ConvTranspose3d(base_channels * 8, base_channels * 4, kernel_size=(2, 2, 2), stride=(2, 2, 2), output_padding=(1, 0, 0))
        self.dec3 = DoubleConv3D(base_channels * 8, base_channels * 4)
        self.up2 = nn.ConvTranspose3d(base_channels * 4, base_channels * 2, kernel_size=(2, 2, 2), stride=(2, 2, 2))
        self.dec2 = DoubleConv3D(base_channels * 4, base_channels * 2)
        self.up1 = nn.ConvTranspose3d(base_channels * 2, base_channels, kernel_size=(2, 2, 2), stride=(2, 2, 2))
        self.dec1 = DoubleConv3D(base_channels * 2, base_channels)
        # 輸出卷積，將通道數降為 1
        self.out_conv = nn.Conv3d(base_channels, 1, kernel_size=1)
        # dropout 用於防止過擬合
        self.dropout = nn.Dropout3d(dropout_rate)
        # 時間嵌入的線性轉換與激活
        self.time_proj = nn.Sequential(nn.Linear(time_emb_dim, base_channels * 8), nn.SiLU())
        # 將完整序列 x_full 通過 1x1 卷積調整通道數，使其與 x_t 保持一致（這裡假設保持 1 通道）
        self.x_full_conv = nn.Conv3d(in_channels, in_channels, kernel_size=1)

    def forward(self, x_t, x_full, t_emb):
        """
        前向傳播函數：
        參數:
            x_t: 含噪聲的部分序列，形狀 (batch, 1, 9, 21, 21)
            x_full: 完整的序列數據，形狀 (batch, 1, 9, 21, 21)
            t_emb: 時間嵌入，形狀 (batch, time_emb_dim)
        回傳:
            模型輸出，形狀 (batch, 1, 1, 21, 21)
        """
        # 處理 x_full，使其通道數與 x_t 一致
        x_full_conv = self.x_full_conv(x_full)  # (batch, 1, 9, 21, 21)
        # 將 x_t 與處理後的 x_full 做融合（逐元素相加）
        x_input = x_t + x_full_conv  # (batch, 1, 9, 21, 21)
        # 編碼器第一層
        e1 = self.enc1(x_input)  # (batch, 64, 9, 21, 21)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        e4 = self.enc4(self.pool3(e3))
        p4 = self.pool4(e4)
        # 將時間嵌入經線性轉換後擴展至與 p4 同維度並與 p4 相加
        t_emb = self.time_proj(t_emb)[:, :, None, None, None]
        b = self.bottleneck(p4 + t_emb)
        b = self.dropout(b)
        # 解碼器：上採樣後與對應編碼層做 concat
        d4 = self.up4(b)
        if d4.shape[-3:] != e4.shape[-3:]:
            # 使用 trilinear 插值調整尺寸
            d4 = F.interpolate(d4, size=e4.shape[-3:], mode='trilinear', align_corners=True)
        d4 = self.dec4(torch.cat([d4, e4], dim=1))
        d3 = self.up3(d4)
        if d3.shape[-3:] != e3.shape[-3:]:
            d3 = F.interpolate(d3, size=e3.shape[-3:], mode='trilinear', align_corners=True)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))
        d2 = self.up2(d3)
        if d2.shape[-3:] != e2.shape[-3:]:
            d2 = F.interpolate(d2, size=e2.shape[-3:], mode='trilinear', align_corners=True)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        d1 = self.up1(d2)
        if d1.shape[-3:] != e1.shape[-3:]:
            d1 = F.interpolate(d1, size=e1.shape[-3:], mode='trilinear', align_corners=True)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))
        # 經過輸出卷積獲得最終結果
        out = self.out_conv(d1)
        # 返回結果，僅保留時間維度上的第一個步驟（1個預測步長）
        return out[:, :, :1, :, :]

class DDPM3D(nn.Module):
    """
    條件式 DDPM (Denoising Diffusion Probabilistic Model) 模型。
    此模型利用前向擴散與反向去噪過程進行生成任務。
    """
    def __init__(self, model: nn.Module, timesteps: int = 1000, 
                 beta_start: float = 1e-4, beta_end: float = 0.02, device: str = 'cuda'):
        super().__init__()
        self.model = model
        self.timesteps = timesteps
        self.device = device
        # 線性生成 beta 值
        self.betas = torch.linspace(beta_start, beta_end, timesteps).to(device)
        self.alphas = 1.0 - self.betas
        # 累乘計算 alpha 的連乘積，用於生成擴散過程的係數
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)
        self.sqrt_alphas_cumprod = torch.sqrt(self.alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - self.alphas_cumprod)
        # 計算頻率因子，用於時間嵌入的正弦與餘弦函數
        self.half_dim = TIME_EMB_DIM // 2
        self.freq_factor = torch.exp(torch.arange(self.half_dim, dtype=torch.float32) *
                                     -(math.log(10000.0) / (self.half_dim - 1))).to(device)

    def get_time_embedding(self, t):
        """
        生成時間嵌入向量，使用正弦與餘弦函數將標量時間映射到向量。
        參數:
            t: 時間步（batch_size,)
        回傳:
            時間嵌入，形狀 (batch_size, TIME_EMB_DIM)
        """
        t = t.float()
        emb = t[:, None] * self.freq_factor.to(t.device)
        return torch.cat([torch.sin(emb), torch.cos(emb)], dim=1)

    def get_condition_embedding(self, cond):
        """
        取得條件嵌入，目前未使用條件資訊，返回全零向量。
        """
        return torch.zeros(cond.shape[0], TIME_EMB_DIM, device=cond.device)

    def q_sample(self, x0, t, noise=None):
        """
        前向擴散過程：根據給定的時間步 t，將數據 x0 擴散成含噪版本。
        參數:
            x0: 原始數據
            t: 時間步（batch_size,)
            noise: 可選噪聲，若未提供則生成隨機噪聲
        回傳:
            擴散後的數據
        """
        if noise is None:
            noise = torch.randn_like(x0)
        # 根據時間步獲取相對應的係數
        sqrt_alpha = self.sqrt_alphas_cumprod[t].view(-1, 1, 1, 1, 1)
        sqrt_one_minus_alpha = self.sqrt_one_minus_alphas_cumprod[t].view(-1, 1, 1, 1, 1)
        # 混合原始數據與噪聲
        return sqrt_alpha * x0 + sqrt_one_minus_alpha * noise

    def p_losses(self, cond, target, t):
        """
        計算去噪損失，目標是讓模型預測出噪聲部分。
        參數:
            cond: 條件序列（含目標數據與前置條件）
            target: 真實目標數據
            t: 時間步
        回傳:
            均方誤差損失
        """
        # 構造完整無噪聲序列：將 target 與條件序列的後半部拼接
        x_full = torch.cat([target, cond[:, :, 1:]], dim=2)
        noise = torch.randn_like(target)
        # 生成含噪 target
        x_noisy_target = self.q_sample(target, t, noise=noise)
        # 將含噪 target 與條件序列拼接，作為模型輸入
        x_t = torch.cat([x_noisy_target, cond[:, :, 1:]], dim=2)
        # 取得時間嵌入與條件嵌入，並融合
        time_emb = self.get_time_embedding(t).to(self.device)
        cond_emb = self.get_condition_embedding(cond)
        combined_emb = time_emb + cond_emb
        # 預測噪聲
        pred_noise = self.model(x_t, x_full, combined_emb)
        pred_noise_target = pred_noise[:, :, :1, :, :]
        # 計算模型預測與實際噪聲間的均方誤差
        return F.mse_loss(pred_noise_target, noise)

    @torch.no_grad()
    def p_sample(self, x_t, t, cond):
        """
        單步反向去噪：從含噪數據 x_t 生成前一步的數據。
        參數:
            x_t: 當前含噪數據，形狀 (batch, 1, 1, H, W) 或 (batch, 1, 9, 21, 21)
            t: 當前時間步
            cond: 條件數據
        回傳:
            去噪後的數據 x_{t-1}
        """
        # 確保 x_t 與 cond 為 5 維張量
        if x_t.dim() == 4:
            x_t = x_t.unsqueeze(1)  # (batch, 1, 1, H, W)
        if cond.dim() == 4:
            cond = cond.unsqueeze(1)  # (batch, 1, 9, 21, 21)
        
        # 取得當前時間步的 beta 等係數
        beta_t = self.betas[t].view(-1, 1, 1, 1, 1)
        sqrt_recip_alpha_t = 1.0 / torch.sqrt(self.alphas[t]).view(-1, 1, 1, 1, 1)
        sqrt_one_minus_alphas_cumprod_t = self.sqrt_one_minus_alphas_cumprod[t].view(-1, 1, 1, 1, 1)
        time_emb = self.get_time_embedding(t).to(self.device)
        cond_emb = self.get_condition_embedding(cond)
        combined_emb = time_emb + cond_emb
        
        # 將含噪數據與條件數據合併
        x_t_full = torch.cat([x_t, cond[:, :, 1:]], dim=2)  # (batch, 1, 9, 21, 21)
        x_full = cond  # 完整序列作為條件
        # 模型預測噪聲
        eps_theta = self.model(x_t_full, x_full, combined_emb)
        eps_theta_target = eps_theta[:, :, :1, :, :]  # 僅取出目標噪聲部分
        
        # 反向去噪步驟：計算 x_{t-1}
        x_t_minus_1 = sqrt_recip_alpha_t * (x_t - beta_t / sqrt_one_minus_alphas_cumprod_t * eps_theta_target)
        # 當 t > 0 時，加入隨機噪聲；t = 0 時直接返回
        mask = (t > 0).float().view(-1, 1, 1, 1, 1)
        sigma_t = torch.sqrt(beta_t)
        noise = torch.randn_like(x_t)
        return x_t_minus_1 + mask * sigma_t * noise

    @torch.no_grad()
    def p_sample_loop(self, shape, cond):
        """
        反向去噪迴圈：從初始純噪聲開始，逐步去噪生成數據。
        參數:
            shape: 生成數據的形狀（可能需要調整為 5D）
            cond: 條件數據
        回傳:
            生成的數據張量
        """
        # 確保 cond 為 5D
        if cond.dim() == 4:
            cond = cond.unsqueeze(1)  # (batch, 1, 9, 21, 21)
        
        # 調整 shape 為 5D
        if len(shape) == 4:
            batch_size = shape[0]
            shape = (batch_size, 1, shape[1], shape[2], shape[3])
        
        # 初始噪聲
        x = torch.randn(shape, device=self.device)
        
        # 由最後一步開始，逐步進行去噪
        for i in reversed(range(self.timesteps)):
            t = torch.full((shape[0],), i, device=self.device, dtype=torch.long)
            x = self.p_sample(x, t, cond)
        
        return x

In [11]:
# --------------------------------------
# 視覺化工具：用於繪製預測結果、誤差網格圖等
# --------------------------------------
def truncate_colormap(cmap, minval: float = 0.0, maxval: float = 1.0, n: int = 256):
    """
    截斷 colormap，僅使用其中一部分的色階範圍。
    參數:
        cmap: 原始的 colormap
        minval, maxval: 取色範圍
        n: 取樣點數
    回傳:
        新的截斷後的 colormap
    """
    new_cmap = mcolors.LinearSegmentedColormap.from_list(
        f'trunc({cmap.name},{minval:.2f},{maxval:.2f})',
        cmap(np.linspace(minval, maxval, n))
    )
    return new_cmap

def visualize_predictions(cond, generated, target, sample_idx: int = 0, 
                         save_dir: str = r"C:\thesis\code\result_ddpm"):
    """
    視覺化預測結果與真實值的比較，包含生成結果、真實數據、以及誤差（MSE 與 MAE）的圖形。
    參數:
        cond: 條件數據
        generated: 生成結果
        target: 真實目標數據
        sample_idx: 指定要視覺化哪個樣本
        save_dir: 圖形存檔的目錄
    """
    os.makedirs(save_dir, exist_ok=True)
    pred_length = generated.shape[2]
    
    # 若 sample_idx 為 None，則計算所有樣本的平均值
    if sample_idx is None:
        generated_avg = torch.mean(generated, dim=(0, 2)).squeeze(0).cpu().numpy()  # (H, W)
        target_avg = torch.mean(target, dim=(0, 2)).squeeze(0).cpu().numpy()        # (H, W)
        
        mse_matrix = (generated_avg - target_avg) ** 2
        mae_matrix = np.abs(generated_avg - target_avg)
        mape_matrix = np.abs((target_avg - generated_avg) / (target_avg + 1e-10)) * 100
        smape_matrix = np.abs(generated_avg - target_avg) / (np.abs(target_avg) + np.abs(generated_avg) + 1e-10) * 100
        
        mse = np.mean(mse_matrix)
        mae = np.mean(mae_matrix)
        mape = np.mean(mape_matrix)
        smape = np.mean(smape_matrix)
        
        # 子圖 1：平均生成結果
        plt.figure(figsize=(6, 6))
        plt.imshow(generated_avg, cmap='viridis')
        plt.colorbar()
        plt.title('avg_generated')
        plt.savefig(os.path.join(save_dir, 'prediction_all_samples_avg_generated.png'), dpi=300)
        plt.close()
        
        # 子圖 2：平均真實值
        plt.figure(figsize=(6, 6))
        plt.imshow(target_avg, cmap='viridis')
        plt.colorbar()
        plt.title('avg_target')
        plt.savefig(os.path.join(save_dir, 'prediction_all_samples_avg_target.png'), dpi=300)
        plt.close()
        
        # 子圖 3：MSE 誤差圖
        plt.figure(figsize=(6, 6))
        plt.imshow(mse_matrix, cmap='hot')
        plt.colorbar()
        plt.title(f'MSE: {mse:.0f}')
        plt.savefig(os.path.join(save_dir, 'prediction_all_samples_avg_mse.png'), dpi=300)
        plt.close()
        
        # 子圖 4：MAE 誤差圖（標示整數）
        plt.figure(figsize=(6, 6))
        plt.imshow(mae_matrix, cmap='hot')
        plt.colorbar()
        for i in range(mae_matrix.shape[0]):
            for j in range(mae_matrix.shape[1]):
                plt.text(j, i, f'{int(round(mae_matrix[i, j]))}', ha='center', va='center', color='white', fontsize=4)
        plt.title(f'MAE: {mae:.0f}')
        plt.savefig(os.path.join(save_dir, 'prediction_all_samples_avg_mae.png'), dpi=300)
        plt.close()
        
        # 子圖 5：MAPE 誤差圖（標示整數）
        plt.figure(figsize=(6, 6))
        plt.imshow(mape_matrix, cmap='hot')
        plt.colorbar()
        for i in range(mape_matrix.shape[0]):
            for j in range(mape_matrix.shape[1]):
                plt.text(j, i, f'{int(round(mape_matrix[i, j]))}', ha='center', va='center', color='white', fontsize=4)
        plt.title(f'MAPE: {mape:.0f}%')
        plt.savefig(os.path.join(save_dir, 'prediction_all_samples_avg_mape.png'), dpi=300)
        plt.close()
        
        # 子圖 6：SMAPE 誤差圖（標示整數）
        plt.figure(figsize=(6, 6))
        plt.imshow(smape_matrix, cmap='hot')
        plt.colorbar()
        for i in range(smape_matrix.shape[0]):
            for j in range(smape_matrix.shape[1]):
                plt.text(j, i, f'{int(round(smape_matrix[i, j]))}', ha='center', va='center', color='white', fontsize=4)
        plt.title(f'SMAPE: {smape:.0f}%')
        plt.savefig(os.path.join(save_dir, 'prediction_all_samples_avg_smape.png'), dpi=300)
        plt.close()
    
    else:
        # 原有單樣本視覺化邏輯（這裡保留，僅更新為包含 MAPE 和 SMAPE）
        for t in range(pred_length):
            plt.figure(figsize=(20, 4))
            
            plt.subplot(1, 5, 1)
            plt.imshow(generated[sample_idx, 0, t].cpu().numpy(), cmap='viridis')
            plt.colorbar()
            plt.title(f'Generated (t={t})')
            
            plt.subplot(1, 5, 2)
            plt.imshow(target[sample_idx, 0, t].cpu().numpy(), cmap='viridis')
            plt.colorbar()
            plt.title(f'True (t={t})')
            
            error_sq = (generated[sample_idx, 0, t].cpu().numpy() - target[sample_idx, 0, t].cpu().numpy()) ** 2
            plt.subplot(1, 5, 3)
            plt.imshow(error_sq, cmap='hot')
            plt.colorbar()
            plt.title(f'MSE (t={t})')
            
            error_abs = np.abs(generated[sample_idx, 0, t].cpu().numpy() - target[sample_idx, 0, t].cpu().numpy())
            plt.subplot(1, 5, 4)
            plt.imshow(error_abs, cmap='hot')
            plt.colorbar()
            plt.title(f'MAE (t={t})')
            
            # 新增 MAPE 子圖
            mape = np.abs((target[sample_idx, 0, t].cpu().numpy() - generated[sample_idx, 0, t].cpu().numpy()) / 
                         (target[sample_idx, 0, t].cpu().numpy() + 1e-10)) * 100
            plt.subplot(1, 5, 5)
            plt.imshow(mape, cmap='hot')
            plt.colorbar()
            plt.title(f'MAPE (t={t})')
            
            plt.suptitle(f'Sample {sample_idx} - Time Step {t}')
            plt.tight_layout(rect=[0, 0, 1, 0.95])
            plt.savefig(os.path.join(save_dir, f'prediction_sample{sample_idx}_t{t}.png'), dpi=300)
            plt.close()

def plot_grid_with_error(sorted_flow_columns: list, H: int, W: int, 
                         mse_matrix: np.ndarray, mae_matrix: np.ndarray, mape_matrix: np.ndarray, 
                         save_dir: str = r"C:\\thesis\\code\\result_ddpm", smape_matrix: np.ndarray = None):
    """
    繪製網格圖，顯示每個網格點的誤差（MSE、MAE 和 MAPE），並將結果存成圖與表格。
    
    Args:
        sorted_flow_columns (list): 經緯度欄位的排序列表。
        H (int): 網格高度。
        W (int): 網格寬度。
        mse_matrix (np.ndarray): 每個網格點的 MSE 矩陣，形狀為 (H, W)。
        mae_matrix (np.ndarray): 每個網格點的 MAE 矩陣，形狀為 (H, W)。
        mape_matrix (np.ndarray): 每個網格點的 MAPE 矩陣，形狀為 (H, W)。
        save_dir (str): 存檔路徑。
        smape_matrix (np.ndarray, optional): 每個網格點的 SMAPE 矩陣，形狀為 (H, W)。預設為 None。
    """
    os.makedirs(save_dir, exist_ok=True)
    
    locations = [parse_lat_lon(col) for col in sorted_flow_columns]
    longitudes, latitudes = zip(*locations)
    
    orig_cmap = plt.get_cmap('OrRd')
    trunc_cmap = truncate_colormap(orig_cmap, 0.3, 1.0)
    
    # 繪製 MSE 網格圖
    plt.figure(figsize=(12, 12))
    scatter = plt.scatter(longitudes, latitudes, c=mse_matrix.flatten(), cmap=trunc_cmap, marker='o')
    plt.colorbar(scatter, label='MSE')
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.title("Grid with MSE")
    plt.grid(True)
    plt.savefig(os.path.join(save_dir, 'plot_grid_with_error_mse.png'), dpi=600, bbox_inches='tight', pad_inches=0.1)
    plt.close()

    # 繪製 MAE 網格圖
    plt.figure(figsize=(12, 12))
    scatter = plt.scatter(longitudes, latitudes, c=mae_matrix.flatten(), cmap=trunc_cmap, marker='o')
    plt.colorbar(scatter, label='MAE')
    for i, (lon, lat) in enumerate(zip(longitudes, latitudes)):
        plt.text(lon, lat, f'{int(round(mae_matrix.flatten()[i]))}', ha='center', va='center', color='black', fontsize=5)
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.title("Grid with MAE")
    plt.grid(True)
    plt.savefig(os.path.join(save_dir, 'plot_grid_with_error_mae.png'), dpi=600, bbox_inches='tight', pad_inches=0.1)
    plt.close()

    plt.figure(figsize=(12, 12))
    scatter = plt.scatter(longitudes, latitudes, c=mae_matrix.flatten(), cmap=trunc_cmap, marker='o')
    plt.colorbar(scatter, label='MAE')
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.title("Grid with MAE")
    plt.grid(True)
    plt.savefig(os.path.join(save_dir, 'plot_grid_with_error_mae_clean.png'), dpi=600, bbox_inches='tight', pad_inches=0.1)
    plt.close()

    # 繪製 MAPE 網格圖
    plt.figure(figsize=(12, 12))
    scatter = plt.scatter(longitudes, latitudes, c=mape_matrix.flatten(), cmap=trunc_cmap, marker='o')
    plt.colorbar(scatter, label='MAPE (%)')
    for i, (lon, lat) in enumerate(zip(longitudes, latitudes)):
        plt.text(lon, lat, f'{int(round(mape_matrix.flatten()[i]))}', ha='center', va='center', color='black', fontsize=7)
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.title("Grid with MAPE")
    plt.grid(True)
    plt.savefig(os.path.join(save_dir, 'plot_grid_with_error_mape.png'), dpi=600, bbox_inches='tight', pad_inches=0.1)
    plt.close()

    plt.figure(figsize=(12, 12))
    scatter = plt.scatter(longitudes, latitudes, c=mape_matrix.flatten(), cmap=trunc_cmap, marker='o')
    plt.colorbar(scatter, label='MAPE (%)')
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.title("Grid with MAPE")
    plt.grid(True)
    plt.savefig(os.path.join(save_dir, 'plot_grid_with_error_mape_clean.png'), dpi=600, bbox_inches='tight', pad_inches=0.1)
    plt.close()

    # 繪製 SMAPE 網格圖（標示整數）
    if smape_matrix is not None:
        plt.figure(figsize=(12, 12))
        scatter = plt.scatter(longitudes, latitudes, c=smape_matrix.flatten(), cmap=trunc_cmap, marker='o')
        plt.colorbar(scatter, label='SMAPE (%)')
        for i, (lon, lat) in enumerate(zip(longitudes, latitudes)):
            plt.text(lon, lat, f'{int(round(smape_matrix.flatten()[i]))}', ha='center', va='center', color='black', fontsize=7)
        plt.xlabel("Longitude")
        plt.ylabel("Latitude")
        plt.title("Grid with SMAPE")
        plt.grid(True)
        plt.savefig(os.path.join(save_dir, 'plot_grid_with_error_smape.png'), dpi=600, bbox_inches='tight', pad_inches=0.1)
        plt.close()

        plt.figure(figsize=(12, 12))
        scatter = plt.scatter(longitudes, latitudes, c=smape_matrix.flatten(), cmap=trunc_cmap, marker='o')
        plt.colorbar(scatter, label='SMAPE (%)')
        plt.xlabel("Longitude")
        plt.ylabel("Latitude")
        plt.title("Grid with SMAPE")
        plt.grid(True)
        plt.savefig(os.path.join(save_dir, 'plot_grid_with_error_smape_clean.png'), dpi=600, bbox_inches='tight', pad_inches=0.1)
        plt.close()

    # 更新表格，新增 SMAPE
    table_data = {
        'Grid Index': [f'[{i},{j}]' for i in range(H) for j in range(W)],
        'Longitude': longitudes,
        'Latitude': latitudes,
        'MSE': mse_matrix.flatten(),
        'MAE': mae_matrix.flatten(),
        'MAPE (%)': mape_matrix.flatten()
    }
    if smape_matrix is not None:
        table_data['SMAPE (%)'] = smape_matrix.flatten()
    
    df = pd.DataFrame(table_data)
    df.to_csv(os.path.join(save_dir, 'mse_mae_mape_smape_per_coordinate.csv'), index=False)
    df.to_excel(os.path.join(save_dir, 'mse_mae_mape_smape_per_coordinate.xlsx'), index=False)

def compute_rgb_mean_std(dataset_for_stats: Dataset, 
                         num_samples_to_use: int,
                         original_data_mean: torch.Tensor, # 用於反正規化 dataset 中樣本的均值
                         original_data_std: torch.Tensor,  # 用於反正規化 dataset 中樣本的標準差
                         global_norm_min: float,      # 用於熱力圖顏色映射的全局最小值
                         global_norm_max: float,      # 用於熱力圖顏色映射的全局最大值
                         device: str = 'cpu'):
    """
    遍歷部分資料集（例如目標網格轉換後的 RGB 熱力圖），
    計算所有圖像每個通道的平均值與標準差。
    圖像生成過程與 evaluate_model 中為 FID 生成圖像的過程保持一致。
    """
    all_pixels = []
    
    # 確保 num_samples_to_use 不超過數據集實際大小
    actual_samples_to_process = min(len(dataset_for_stats), num_samples_to_use)
    
    # 如果需要，可以隨機抽樣，或者按順序取前 N 個樣本
    # 為了穩定性，通常按順序取或者固定抽樣種子
    sample_indices = range(actual_samples_to_process) 
    # 如果希望隨機抽樣:
    # if actual_samples_to_process < len(dataset_for_stats):
    #     sample_indices = random.sample(range(len(dataset_for_stats)), actual_samples_to_process)
    # else:
    #     sample_indices = range(actual_samples_to_process)

    # original_data_mean 和 original_data_std 應該已經在正確的 device 上 (從 evaluate_model 傳入)
    original_data_mean = original_data_mean.to(device)
    original_data_std = original_data_std.to(device)

    for idx in sample_indices:
        # 這裡以目標數據為例，shape 可能是 (1, 1, H, W) 或 (1, H, W) 等
        # dataset_for_stats 通常是 test_dataset，其 __getitem__ 返回 (cond, target)
        # target 是已經經過 Z-score 正規化的數據
        _, target_normalized = dataset_for_stats[idx] 
        
        target_normalized = target_normalized.to(device) # 確保在正確的 device
        
        # 1. 反正規化 target (使用原始訓練數據集的 mean 和 std)
        # target_normalized 可能有額外的批次或通道維度，使用 squeeze()
        # 假設 target_normalized 的形狀是 (1, 1, H, W) 或 (1, H, W)
        # 我們需要的是 (H, W) 的 grid
        if target_normalized.dim() == 4: # (batch, channel, H, W) -> 通常是 (1,1,H,W)
            grid_squeezed = target_normalized.squeeze(0).squeeze(0) 
        elif target_normalized.dim() == 3: # (channel, H, W) -> 通常是 (1,H,W) for __getitem__
            grid_squeezed = target_normalized.squeeze(0)
        elif target_normalized.dim() == 2: # (H,W)
            grid_squeezed = target_normalized
        else:
            raise ValueError(f"Unsupported target_normalized shape: {target_normalized.shape}")

        grid_denormalized_tensor = grid_squeezed * original_data_std + original_data_mean
        grid_denormalized = grid_denormalized_tensor.cpu().numpy() # (H, W)
        
        # 2. 使用 evaluate_model 中的 global_norm_min 和 global_norm_max 進行正規化以生成熱力圖
        # 這與 evaluate_model 中生成 pred_images/real_images 的方式一致
        norm_grid_for_rgb = (grid_denormalized - global_norm_min) / (global_norm_max - global_norm_min + 1e-8)
        norm_grid_for_rgb = np.clip(norm_grid_for_rgb, 0, 1) # 確保值在 [0, 1] 範圍內
        
        # 使用 viridis colormap，得到 (H, W, 4) RGBA，再取前三個通道
        rgb = (plt.cm.viridis(norm_grid_for_rgb)[..., :3] * 255).astype(np.uint8)
        rgb_normalized_for_inception = rgb.astype(np.float32) / 255.0 # 像素值正規化到 [0,1]
        all_pixels.append(rgb_normalized_for_inception.reshape(-1, 3)) # (H*W, 3)
    
    if not all_pixels:
        # 如果沒有處理任何樣本（例如 num_samples_to_use=0 或 dataset_for_stats 為空）
        # 返回 InceptionV3 常用均值標準差作為後備，或者拋出錯誤
        # ImageNet 均值/標準差約為 [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
        logging.warning("No pixels processed in compute_rgb_mean_std. Returning default ImageNet stats.")
        return [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

    all_pixels_np = np.concatenate(all_pixels, axis=0)
    mean_rgb = np.mean(all_pixels_np, axis=0)
    std_rgb = np.std(all_pixels_np, axis=0)
    
    return mean_rgb.tolist(), std_rgb.tolist()


In [12]:
# 訓練
def train_ddpm(diffusion: DDPM3D, train_loader: DataLoader, val_loader: DataLoader, 
               epochs: int = 20, lr: float = 1e-4, device: str = 'cuda', 
               patience: int = 3, weight_decay: float = 1e-6, 
               save_dir: str = r"C:\thesis\code\result_ddpm",
               checkpoint_interval: int = 5) -> DDPM3D:
    """
    訓練 DDPM 模型，並進行驗證與早停檢查，加入動態學習率調整與學習率記錄。
    
    Args:
        diffusion: DDPM 模型實例
        train_loader, val_loader: 訓練與驗證的 DataLoader
        epochs: 最大訓練輪數
        lr: 初始學習率
        device: 訓練設備，例如 'cuda' 或 'cpu'
        patience: 早停耐心次數
        weight_decay: 優化器的權重衰減
        save_dir: 模型與結果的存檔目錄
        checkpoint_interval: 檢查點保存間隔（每隔多少 epoch 保存一次檢查點）
    
    Returns:
        訓練後的 diffusion 模型
    """
    optimizer = optim.AdamW(diffusion.parameters(), lr=lr, weight_decay=weight_decay)
    # 添加學習率調度器：當驗證損失不再下降時，將學習率乘以 0.5，最小學習率為 1e-6
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-6)
    diffusion.to(device)
    best_val_loss = float('inf')
    patience_counter = 0
    train_losses, val_losses = [], []
    lr_history = []  # 用於記錄每個 epoch 的學習率

    os.makedirs(save_dir, exist_ok=True)
    checkpoint_path = os.path.join(save_dir, 'checkpoint.pth')

    # # 檢查並恢復檢查點
    # if os.path.exists(checkpoint_path):
    #     checkpoint = torch.load(checkpoint_path)
    #     diffusion.load_state_dict(checkpoint['model_state_dict'])
    #     optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    #     # 檢查是否有 scheduler_state_dict，若無則重新初始化 scheduler
    #     if 'scheduler_state_dict' in checkpoint:
    #         scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    #         logging.info("恢復 scheduler 狀態")
    #     else:
    #         logging.warning("檢查點中未找到 'scheduler_state_dict'，使用默認 scheduler 設置")
    #         if 'learning_rate' in checkpoint:
    #             current_lr = checkpoint['learning_rate']
    #             for param_group in optimizer.param_groups:
    #                 param_group['lr'] = current_lr
    #             logging.info(f"從檢查點恢復學習率: {current_lr:.8f}")
        
    #     start_epoch = checkpoint['epoch']
    #     train_losses = checkpoint['train_losses']
    #     val_losses = checkpoint['val_losses']
    #     best_val_loss = min(val_losses) if val_losses else float('inf')
    #     logging.info(f"恢復訓練，從 epoch {start_epoch} 開始")
    # else:
    start_epoch = 0

    for epoch in range(start_epoch, epochs):
        diffusion.train()
        total_train_loss = 0
        # 逐批訓練
        for cond, target in train_loader:
            cond, target = cond.to(device), target.to(device)
            optimizer.zero_grad()
            # 隨機抽取一個時間步
            t = torch.randint(0, diffusion.timesteps, (target.shape[0],), device=device)
            loss = diffusion.p_losses(cond, target, t)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(diffusion.parameters(), max_norm=1.0)
            optimizer.step()
            total_train_loss += loss.item()
        
        avg_train_loss = total_train_loss / len(train_loader)
        train_losses.append(avg_train_loss)

        # 驗證模式
        diffusion.eval()
        total_val_loss = 0
        with torch.no_grad():
            for cond, target in val_loader:
                cond, target = cond.to(device), target.to(device)
                t = torch.randint(0, diffusion.timesteps, (target.shape[0],), device=device)
                loss = diffusion.p_losses(cond, target, t)
                total_val_loss += loss.item()
        
        avg_val_loss = total_val_loss / len(val_loader)
        val_losses.append(avg_val_loss)

        # 更新學習率
        scheduler.step(avg_val_loss)
        current_lr = scheduler.get_last_lr()[0]  # 獲取當前學習率
        lr_history.append(current_lr)  # 記錄學習率

        logging.info(f"Epoch [{epoch+1}/{epochs}] - Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}, Learning Rate: {current_lr:.8f}")

        # 若驗證損失降低則儲存最佳模型，否則耐心計數增加
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
            torch.save({
                'model_state_dict': diffusion.state_dict(),
                'learning_rate': current_lr,  # 記錄最佳模型的學習率
            }, os.path.join(save_dir, 'best_model.pth'))
            logging.info(f"保存最佳模型，驗證損失: {best_val_loss:.4f}, 學習率: {current_lr:.8f}")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                logging.info("Early stopping triggered.")
                break

        # 定期保存檢查點
        if (epoch + 1) % checkpoint_interval == 0:
            torch.save({
                'model_state_dict': diffusion.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'epoch': epoch + 1,
                'train_losses': train_losses,
                'val_losses': val_losses,
                'learning_rate': current_lr,
            }, checkpoint_path)
            logging.info(f"在 epoch {epoch + 1} 保存檢查點，學習率: {current_lr:.8f}")

    # 繪製訓練與驗證損失曲線，並存檔
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(train_losses) + 1), train_losses, label='Train Loss')
    plt.plot(range(1, len(val_losses) + 1), val_losses, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(save_dir, 'loss_curve.png'), dpi=300, bbox_inches='tight')
    plt.close()

    # 繪製學習率曲線，並存檔
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(lr_history) + 1), lr_history, label='Learning Rate')
    plt.xlabel('Epoch')
    plt.ylabel('Learning Rate')
    plt.title('Learning Rate Curve')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(save_dir, 'lr_curve.png'), dpi=300, bbox_inches='tight')
    plt.close()

    return diffusion


In [13]:
# --------------------------------------
# 評估函數
# --------------------------------------

@torch.no_grad()
def evaluate_model(diffusion: DDPM3D, dataset: Dataset, device: str = 'cuda', 
                   max_samples: int = 100, save_dir: str = r"C:\\thesis\\code\\result_ddpm",
                   sample_idx: int = 0) -> dict:
    diffusion.eval()
    metrics = {'mse': 0.0, 'mae': 0.0, 'mape': 0.0, 'smape': 0.0, 'fid': 0.0}
    N = min(len(dataset), max_samples) # 實際使用的樣本數
    if N == 0:
        logging.warning("No samples to evaluate. Returning empty metrics.")
        return metrics
        
    sample_indices = random.sample(range(len(dataset)), N)
    
    base_dataset = dataset.dataset if isinstance(dataset, Subset) else dataset
    H, W = base_dataset.H, base_dataset.W
    pred_length = base_dataset.prediction_length # 假設 prediction_length 是 1
    
    # 獲取用於反正規化的原始數據集均值和標準差
    # 這些值應該是在 PeopleFlowDatasetCondition 初始化時計算的全局值
    original_dataset_mean = base_dataset.mean_val.to(device)
    original_dataset_std = base_dataset.std_val.to(device)
    
    generated_batch = torch.zeros(N, 1, pred_length, H, W, device=device)
    target_batch = torch.zeros(N, 1, pred_length, H, W, device=device)
    
    for i, current_idx in enumerate(sample_indices): # 使用 current_idx 避免與 visualize_predictions 的 sample_idx 混淆
        cond, target_normalized = dataset[current_idx] # target_normalized 是 Z-score 正規化的
        cond, target_normalized = cond.to(device), target_normalized.to(device)
        
        # 反正規化 target 以得到 target_original，用於後續比較和 FID 的真實圖像
        # 確保 target_normalized 有正確的維度以進行廣播
        # 假設 target_normalized 從 dataset 來是 (1, H, W) 或 (1, 1, H, W)
        # 而 original_dataset_std/mean 是標量或可以廣播的形狀
        _target_squeezed = target_normalized.squeeze(0) if target_normalized.shape[0] == 1 else target_normalized
        target_original_single = _target_squeezed * original_dataset_std + original_dataset_mean 
        # target_original_single 的形狀應為 (1, H, W) 或 (C, H, W)
        # 我們需要 (1, pred_length, H, W) for target_batch
        if target_original_single.dim() == 2: # (H,W) -> (1,1,H,W)
            target_original_single = target_original_single.unsqueeze(0).unsqueeze(0)
        elif target_original_single.dim() == 3: # (C,H,W) -> (1,C,H,W)
             target_original_single = target_original_single.unsqueeze(0)
        # 確保通道數為1，pred_length 為1 (基於目前代碼的普遍假設)
        target_batch[i] = target_original_single[:, :pred_length, :, :]


        # 生成預測 x_recon (模型輸出的是正規化尺度的)
        # p_sample_loop 期望的 shape 是 (batch, channels, depth, H, W)
        # target_normalized 此時可能是 (1,1,H,W) or (1,H,W)
        # 我們需要為 p_sample_loop 提供一個與其輸出一致的 shape
        # DDPM 輸出 x_recon 是正規化尺度的
        # DDPM.p_sample_loop takes shape of the *target* (denoised, but on normalized scale)
        # The shape for p_sample_loop should be (batch_size, channels, depth/sequence, H, W)
        # For a single prediction step, depth/sequence is pred_length
        # Let's assume pred_length = 1, channels = 1
        shape_for_sampling = (1, 1, pred_length, H, W) # batch_size is 1 for single sample generation
        x_recon_normalized = diffusion.p_sample_loop(shape_for_sampling, cond) # x_recon_normalized on normalized scale

        # 反正規化 x_recon
        x_recon_original_single = x_recon_normalized * original_dataset_std + original_dataset_mean
        generated_batch[i] = x_recon_original_single
        
        # 計算指標時使用反正規化後的值
        # 將 x_recon_original_single 的第3個維度 (索引為2) 壓縮掉，使其形狀與 target 匹配
        x_recon_squeezed = x_recon_original_single.squeeze(2) # 新增這行，或直接在下面使用 .squeeze(2)

        _target_for_metric = target_original_single[:,:pred_length,:,:] # 這個的形狀是 [1, 1, 21, 21]

        mse = F.mse_loss(x_recon_squeezed, _target_for_metric).item()
        mae = F.l1_loss(x_recon_squeezed, _target_for_metric).item()
        # MAPE 和 SMAPE 計算時避免除以零
        mape_val = torch.mean(torch.abs((_target_for_metric - x_recon_squeezed) / (_target_for_metric + 1e-10))) * 100
        smape_val = torch.mean(torch.abs(x_recon_squeezed - _target_for_metric) / \
                              (torch.abs(_target_for_metric) + torch.abs(x_recon_squeezed) + 1e-10)) * 100
        
        metrics['mse'] += mse
        metrics['mae'] += mae
        metrics['mape'] += mape_val.item()
        metrics['smape'] += smape_val.item()
    
    metrics['mse'] /= N
    metrics['mae'] /= N
    metrics['mape'] /= N
    metrics['smape'] /= N

    # 轉換為 RGB 熱力圖，並儲存圖檔
    os.makedirs(save_dir, exist_ok=True)
    pred_images_for_fid = [] # 更名以區分
    real_images_for_fid = [] # 更名以區分

    # 計算全局範圍以統一色階 (基於反正規化後的數據)
    all_pred_flat = generated_batch.cpu().numpy().flatten()
    all_real_flat = target_batch.cpu().numpy().flatten()
    # 處理可能為空的情況 (如果 N=0)
    if all_pred_flat.size == 0 and all_real_flat.size == 0:
        logging.warning("No data for global min/max calculation. FID might be unreliable.")
        global_min, global_max = 0, 1 # Default if no data
    else:
        valid_preds = all_pred_flat[np.isfinite(all_pred_flat)]
        valid_reals = all_real_flat[np.isfinite(all_real_flat)]
        if valid_preds.size == 0 and valid_reals.size == 0:
            logging.warning("All prediction/real data is non-finite. FID might be unreliable.")
            global_min, global_max = 0,1
        else:
            min_pred = valid_preds.min() if valid_preds.size > 0 else np.inf
            max_pred = valid_preds.max() if valid_preds.size > 0 else -np.inf
            min_real = valid_reals.min() if valid_reals.size > 0 else np.inf
            max_real = valid_reals.max() if valid_reals.size > 0 else -np.inf
            global_min = min(min_pred, min_real)
            global_max = max(max_pred, max_real)
            if global_min == global_max: # 防止除以零
                global_max += 1e-8


    for i in range(N):
        for t_step in range(pred_length): # 迭代預測序列的每個時間步
            pred_arr = generated_batch[i, 0, t_step].cpu().numpy()  # (H, W) - 假設 channel 0
            real_arr = target_batch[i, 0, t_step].cpu().numpy()     # (H, W) - 假設 channel 0
            
            # 使用全局範圍正規化到 [0, 1] 以生成熱力圖
            pred_norm_for_rgb = (pred_arr - global_min) / (global_max - global_min + 1e-8)
            real_norm_for_rgb = (real_arr - global_min) / (global_max - global_min + 1e-8)
            pred_norm_for_rgb = np.clip(pred_norm_for_rgb, 0, 1)
            real_norm_for_rgb = np.clip(real_norm_for_rgb, 0, 1)
            
            # ... (繪製並排熱力圖的代碼，與之前類似，使用 pred_norm_for_rgb, real_norm_for_rgb) ...
            # (此處省略詳細繪圖代碼以簡潔，假設它正確使用上述正規化值)
            # 例如:
            # fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8), sharey=True)
            # im1 = ax1.imshow(pred_norm_for_rgb, cmap='viridis', interpolation='bilinear')
            # ax1.set_title(f"Predicted (Sample {sample_indices[i]}, t={t_step})")
            # im2 = ax2.imshow(real_norm_for_rgb, cmap='viridis', interpolation='bilinear')
            # ax2.set_title(f"Real (Sample {sample_indices[i]}, t={t_step})")
            # fig.colorbar(im1, ax=[ax1, ax2], orientation='vertical', label='Normalized Value')
            # combined_filename = os.path.join(save_dir, f"sample{sample_indices[i]}_t{t_step}_heatmap_combined.png")
            # plt.savefig(combined_filename, dpi=300, bbox_inches='tight')
            # plt.close(fig)


            # 生成獨立的 RGB 圖像以供 FID 計算
            pred_rgb = (plt.cm.viridis(pred_norm_for_rgb)[..., :3] * 255).astype(np.uint8)
            real_rgb = (plt.cm.viridis(real_norm_for_rgb)[..., :3] * 255).astype(np.uint8)
            
            pred_images_for_fid.append(Image.fromarray(pred_rgb))
            real_images_for_fid.append(Image.fromarray(real_rgb))
    
    # 在特徵提取前檢查樣本數
    logging.info(f"Number of pred_images_for_fid: {len(pred_images_for_fid)}")
    if len(pred_images_for_fid) < 2 or len(real_images_for_fid) < 2:
        logging.warning("樣本數不足以計算 FID (至少需要 2 個樣本)，將 FID 設為 NaN。")
        metrics['fid'] = float('nan')
    else:
        # 使用修正後的 compute_rgb_mean_std
        # original_dataset_mean 和 original_dataset_std 已經在上面定義並移至 device
        new_mean_rgb, new_std_rgb = compute_rgb_mean_std(
            dataset_for_stats=dataset, # 使用傳入的 test_dataset
            num_samples_to_use=N,      # 使用與評估相同的樣本數
            original_data_mean=original_dataset_mean,
            original_data_std=original_dataset_std,
            global_norm_min=global_min,
            global_norm_max=global_max,
            device=device
        )
        
        inception_model = inception_v3(weights=Inception_V3_Weights.IMAGENET1K_V1, aux_logits=True)
        inception_model.fc = torch.nn.Identity() 
        if hasattr(inception_model, 'AuxLogits'): # 檢查是否存在 AuxLogits
            inception_model.AuxLogits = None 
        inception_model.to(device)
        inception_model.eval()
        
        inception_transform = transforms.Compose([
            transforms.Resize((299, 299)),
            transforms.ToTensor(), # 將 PIL Image (0-255, H,W,C) 轉為 (C,H,W) Tensor (0.0-1.0)
            transforms.Normalize(new_mean_rgb, new_std_rgb) # 使用新計算的 RGB 均值標準差
        ])
        
        # pred_tensors = torch.stack([inception_transform(img).to(device) for img in pred_images_for_fid]) # to(device) in list comprehension
        # real_tensors = torch.stack([inception_transform(img).to(device) for img in real_images_for_fid])
        pred_tensors_list = []
        for img in pred_images_for_fid:
            pred_tensors_list.append(inception_transform(img))
        pred_tensor_batch = torch.stack(pred_tensors_list).to(device)

        real_tensors_list = []
        for img in real_images_for_fid:
            real_tensors_list.append(inception_transform(img))
        real_tensor_batch = torch.stack(real_tensors_list).to(device)

        with torch.no_grad():
            pred_features = inception_model(pred_tensor_batch)
            real_features = inception_model(real_tensor_batch)
        
        pred_features_np = pred_features.cpu().numpy()
        real_features_np = real_features.cpu().numpy()

        mu_pred = np.mean(pred_features_np, axis=0)
        mu_real = np.mean(real_features_np, axis=0)
        sigma_pred = np.cov(pred_features_np, rowvar=False)
        sigma_real = np.cov(real_features_np, rowvar=False)
        
        # 添加 epsilon 以提高協方差矩陣的數值穩定性
        epsilon = 1e-6
        sigma_pred_stable = sigma_pred + np.eye(sigma_pred.shape[0]) * epsilon
        sigma_real_stable = sigma_real + np.eye(sigma_real.shape[0]) * epsilon
        
        covmean, _ = scipy.linalg.sqrtm(sigma_pred_stable.dot(sigma_real_stable), disp=False)
        if np.iscomplexobj(covmean):
            covmean = covmean.real
        
        # 檢查 covmean 是否全為零或 NaN (可能由奇異協方差矩陣導致)
        if np.allclose(covmean, 0) or np.isnan(covmean).any():
            logging.warning("sqrtm(sigma_pred.dot(sigma_real)) resulted in zero or NaN matrix. FID may be inaccurate.")
            # 在這種情況下，可以選擇不計算 trace(2 * covmean) 部分或將其設為0
            trace_covmean = 0
        else:
            trace_covmean = np.trace(covmean)

        fid = np.sum((mu_pred - mu_real)**2) + np.trace(sigma_pred_stable) + np.trace(sigma_real_stable) - 2 * trace_covmean
        metrics['fid'] = fid
    
    # ... (後續的誤差矩陣計算、儲存表格和視覺化的程式碼) ...
    # 注意: evaluate_model 中的 sample_idx 是用於 visualize_predictions 的，
    # FID 計算和總體指標是基於 N 個隨機樣本。
    
    # 更新: 確保 plot_grid_with_error 和 visualize_predictions 使用正確的數據
    # (這部分未作修改，假設它們的輸入是正確的)
    
    # 計算每個網格點的誤差矩陣 (基於反正規化後的數據)
    error_matrix_mse_batch = (generated_batch - target_batch) ** 2 # generated_batch 和 target_batch 都是 (N, 1, pred_length, H, W)
    mse_matrix_grid = torch.mean(error_matrix_mse_batch, dim=(0, 1, 2)).cpu().numpy()  # (H, W), 平均所有樣本、通道和預測長度

    error_matrix_mae_batch = torch.abs(generated_batch - target_batch)
    mae_matrix_grid = torch.mean(error_matrix_mae_batch, dim=(0, 1, 2)).cpu().numpy()  # (H, W)

    # MAPE 和 SMAPE 的 per-grid 計算需要小心處理維度
    # target_batch 和 generated_batch 都是 (N, 1, pred_length, H, W)
    mape_matrix_grid_batch = torch.abs((target_batch - generated_batch) / (target_batch + 1e-10)) * 100
    mape_matrix_grid = torch.mean(mape_matrix_grid_batch, dim=(0, 1, 2)).cpu().numpy() # (H, W)

    smape_matrix_grid_batch = torch.abs(generated_batch - target_batch) / (torch.abs(target_batch) + torch.abs(generated_batch) + 1e-10) * 100
    smape_matrix_grid = torch.mean(smape_matrix_grid_batch, dim=(0, 1, 2)).cpu().numpy() # (H, W)
    
    # 視覺化 (傳入 grid-wise error matrices)
    plot_grid_with_error(base_dataset.sorted_flow_columns, H, W, 
                         mse_matrix_grid, mae_matrix_grid, mape_matrix_grid, 
                         save_dir, smape_matrix_grid)
    
    # visualize_predictions 的 sample_idx 參數是從 evaluate_model 的參數傳入的
    # 它用於選擇 generated_batch 和 target_batch 中的特定樣本進行視覺化
    # 確保 sample_idx 在 [0, N-1] 範圍內
    if N > 0 and sample_idx < N :
         # 傳入的 sample_idx 是相對於 N 個抽樣樣本的索引
        visualize_predictions(None, generated_batch, target_batch, sample_idx=sample_idx, save_dir=save_dir)
    elif N > 0: # 如果 sample_idx 超出範圍，則默認顯示第一個樣本
        logging.warning(f"sample_idx {sample_idx} is out of range for {N} samples. Visualizing sample 0 instead.")
        visualize_predictions(None, generated_batch, target_batch, sample_idx=0, save_dir=save_dir)

    # ... (儲存評估指標到 txt 和 json 的程式碼，記得將硬編碼的樣本數 20 改為 N) ...
    # 例如:
    with open(os.path.join(save_dir, 'evaluation_metrics.txt'), 'w') as f:
        f.write(f"Evaluation Metrics (computed on {N} samples):\n") # 使用 N
        # ... 其他指標 ...
        f.write(f"Reconstruction FID: {metrics['fid']:.6f}\n")

    with open(os.path.join(save_dir, 'evaluation_metrics.json'), 'w') as f:
        json.dump({
            "mse": metrics['mse'], 
            "mae": metrics['mae'], 
            "mape": metrics['mape'], 
            "smape": metrics['smape'],
            "fid": metrics['fid'],
            "sample_size": N, # 使用 N
            "timestamp": pd.Timestamp.now().isoformat()
        }, f, indent=4)

    return metrics

In [14]:
# --------------------------------------
# 主程式進入點
# --------------------------------------
if __name__ == "__main__":
    # -------------------------------
    # 參數設定：網格尺寸、序列長度、批次大小、訓練輪數等
    # -------------------------------
    H, W = 21, 21
    condition_length, prediction_length = 8, 1
    batch_size, epochs, lr, timesteps, patience = 150, 150, 0.0015, 1500, 15
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    save_dir = r"C:\thesis\code\result_ddpm"
    checkpoint_interval = 5  

    # 設定隨機種子，確保實驗結果可重現
    torch.manual_seed(42)
    np.random.seed(42)

    # -------------------------------
    # 初始化數據集，並依比例劃分訓練、驗證與測試集
    # -------------------------------
    dataset = PeopleFlowDatasetCondition(
        csv_path=r"C:\thesis\code\Taipei_CF\all_merged.csv",
        H=H, W=W, condition_length=condition_length, prediction_length=prediction_length,
        normalize=True, debug=True
    )
    train_end = int(0.7 * len(dataset))
    val_end = int(0.85 * len(dataset))
    train_dataset = Subset(dataset, range(0, train_end))
    val_dataset = Subset(dataset, range(train_end, val_end))
    test_dataset = Subset(dataset, range(val_end, len(dataset)))

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

    # -------------------------------
    # 初始化模型
    # 使用 UNet3D 作為去噪模型，並建立 DDPM3D 實例
    # -------------------------------
    unet = UNet3D(in_channels=1, base_channels=64, time_emb_dim=TIME_EMB_DIM, dropout_rate=0.2)
    diffusion = DDPM3D(model=unet, timesteps=timesteps, beta_start=1e-4, beta_end=0.02, device=device)

訊息: 座標點數量 (495) 多於網格數 (441). 將選擇最靠近地理中心的 441 個座標點進行映射。


In [15]:
# -------------------------------
# 訓練模型
# -------------------------------
trained_diffusion = train_ddpm(diffusion, train_loader, val_loader, epochs=epochs, 
                               lr=lr, device=device, patience=patience, save_dir=save_dir,
                               checkpoint_interval=checkpoint_interval)

INFO:root:Epoch [1/150] - Train Loss: 0.3301, Val Loss: 0.1243, Learning Rate: 0.00150000
INFO:root:保存最佳模型，驗證損失: 0.1243, 學習率: 0.00150000
INFO:root:Epoch [2/150] - Train Loss: 0.0831, Val Loss: 0.0875, Learning Rate: 0.00150000
INFO:root:保存最佳模型，驗證損失: 0.0875, 學習率: 0.00150000
INFO:root:Epoch [3/150] - Train Loss: 0.0604, Val Loss: 0.0526, Learning Rate: 0.00150000
INFO:root:保存最佳模型，驗證損失: 0.0526, 學習率: 0.00150000
INFO:root:Epoch [4/150] - Train Loss: 0.0503, Val Loss: 0.0457, Learning Rate: 0.00150000
INFO:root:保存最佳模型，驗證損失: 0.0457, 學習率: 0.00150000
INFO:root:Epoch [5/150] - Train Loss: 0.0451, Val Loss: 0.0489, Learning Rate: 0.00150000
INFO:root:在 epoch 5 保存檢查點，學習率: 0.00150000
INFO:root:Epoch [6/150] - Train Loss: 0.0384, Val Loss: 0.0382, Learning Rate: 0.00150000
INFO:root:保存最佳模型，驗證損失: 0.0382, 學習率: 0.00150000
INFO:root:Epoch [7/150] - Train Loss: 0.0384, Val Loss: 0.0354, Learning Rate: 0.00150000
INFO:root:保存最佳模型，驗證損失: 0.0354, 學習率: 0.00150000
INFO:root:Epoch [8/150] - Train Loss: 0.0344, 

In [18]:
# -------------------------------
# 評估模型
# -------------------------------
max_samples = 360
metrics = evaluate_model(trained_diffusion, test_dataset, device=device, max_samples=max_samples, save_dir=save_dir)
# 更新 logging.info，新增 FID 輸出
logging.info(f"Reconstruction MSE: {metrics['mse']:.6f}, MAE: {metrics['mae']:.6f}, "
             f"MAPE: {metrics['mape']:.6f}, SMAPE: {metrics['smape']:.6f}, "
             f"FID: {metrics['fid']:.6f}")

# 儲存最終評估結果
os.makedirs(save_dir, exist_ok=True)
with open(os.path.join(save_dir, 'evaluation_metrics.txt'), 'w') as f:
    # 修正樣本數，使用實際的 max_samples 值
    f.write(f"Evaluation Metrics (computed on {max_samples} samples):\n")
    f.write(f"Date: {pd.Timestamp.now()}\n")
    f.write(f"Reconstruction MSE: {metrics['mse']:.6f}\n")
    f.write(f"Reconstruction MAE: {metrics['mae']:.6f}\n")
    f.write(f"Reconstruction MAPE: {metrics['mape']:.6f}%\n")
    f.write(f"Reconstruction SMAPE: {metrics['smape']:.6f}%\n")
    # 新增 FID 記錄
    f.write(f"Reconstruction FID: {metrics['fid']:.6f}\n")

with open(os.path.join(save_dir, 'evaluation_metrics.json'), 'w') as f:
    # 更新 JSON 檔案，新增 FID
    json.dump({
        "mse": metrics['mse'], 
        "mae": metrics['mae'], 
        "mape": metrics['mape'], 
        "smape": metrics['smape'],
        "fid": metrics['fid'],  # 新增 FID
        "sample_size": max_samples, 
        "timestamp": pd.Timestamp.now().isoformat()
    }, f, indent=4)

INFO:root:Number of pred_images_for_fid: 360
INFO:root:Reconstruction MSE: 1845396.769645, MAE: 668.935316, MAPE: 40.277618, SMAPE: 15.604890, FID: 4.004450
